In [158]:
from pyspark.sql import SparkSession, column
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, IntegerType, DoubleType, StructType, StructField, DateType, DecimalType, LongType, BooleanType, TimestampType, FloatType, NumericType

In [159]:
# Sessão Spark conectada ao MinIO (s3a)
spark = (
    SparkSession.builder.appName("silver.py")
    .master("spark://spark-master:7077")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "minio")
    .config("spark.hadoop.fs.s3a.secret.key", "minio123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.fast.upload", "true")
    .config("spark.driver.extraClassPath", "/home/jovyan/extra-jars/*")
    .config("spark.executor.extraClassPath", "/opt/spark/extra-jars/*")
    .getOrCreate()
)

In [160]:
spark.getActiveSession()

In [161]:
# Leitura da camada bronze
df_original = spark.read.csv("s3a://bronze/kc_house_data.csv", header=True)

In [162]:
df_original.printSchema()

root
 |-- id: string (nullable = true)
 |-- date: string (nullable = true)
 |-- price: string (nullable = true)
 |-- bedrooms: string (nullable = true)
 |-- bathrooms: string (nullable = true)
 |-- sqft_living: string (nullable = true)
 |-- sqft_lot: string (nullable = true)
 |-- floors: string (nullable = true)
 |-- waterfront: string (nullable = true)
 |-- view: string (nullable = true)
 |-- condition: string (nullable = true)
 |-- grade: string (nullable = true)
 |-- sqft_above: string (nullable = true)
 |-- sqft_basement: string (nullable = true)
 |-- yr_built: string (nullable = true)
 |-- yr_renovated: string (nullable = true)
 |-- zipcode: string (nullable = true)
 |-- lat: string (nullable = true)
 |-- long: string (nullable = true)
 |-- sqft_living15: string (nullable = true)
 |-- sqft_lot15: string (nullable = true)



In [163]:
df_original.show(10)

+----------+---------------+----------+--------+---------+-----------+--------+------+----------+----+---------+-----+----------+-------------+--------+------------+-------+-------+--------+-------------+----------+
|        id|           date|     price|bedrooms|bathrooms|sqft_living|sqft_lot|floors|waterfront|view|condition|grade|sqft_above|sqft_basement|yr_built|yr_renovated|zipcode|    lat|    long|sqft_living15|sqft_lot15|
+----------+---------------+----------+--------+---------+-----------+--------+------+----------+----+---------+-----+----------+-------------+--------+------------+-------+-------+--------+-------------+----------+
|7129300520|20141013T000000|    221900|       3|        1|       1180|    5650|     1|         0|   0|        3|    7|      1180|            0|    1955|           0|  98178|47.5112|-122.257|         1340|      5650|
|6414100192|20141209T000000|    538000|       3|     2.25|       2570|    7242|     2|         0|   0|        3|    7|      2170|       

In [164]:
# Base de trabalho e remoção de espaços em branco
df_columns = df_original.columns
df_sem_espacos = df_original

# Remoção de espaços -> ""
for c in df_columns:
    df_sem_espacos = df_sem_espacos.withColumn(c, F.regexp_replace(c, r"\s+", ""))

# Quantidade de registros com string vazia ("")
df_sem_espacos.select([
    F.sum((F.col(c) == "").cast("int")).alias(c)
    for c in df_columns
]).show()

# Quantidade de registros NULL / None
df_sem_espacos.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_columns
]).show()

# Validação de duplicadas
df_sem_espacos.groupBy(df_columns).count().filter(F.col("count") > 1).count()

+---+----+-----+--------+---------+-----------+--------+------+----------+----+---------+-----+----------+-------------+--------+------------+-------+---+----+-------------+----------+
| id|date|price|bedrooms|bathrooms|sqft_living|sqft_lot|floors|waterfront|view|condition|grade|sqft_above|sqft_basement|yr_built|yr_renovated|zipcode|lat|long|sqft_living15|sqft_lot15|
+---+----+-----+--------+---------+-----------+--------+------+----------+----+---------+-----+----------+-------------+--------+------------+-------+---+----+-------------+----------+
|  0|   0|    0|       0|        0|          0|       0|     0|         0|   0|        0|    0|         0|            0|       0|           0|      0|  0|   0|            0|         0|
+---+----+-----+--------+---------+-----------+--------+------+----------+----+---------+-----+----------+-------------+--------+------------+-------+---+----+-------------+----------+

+---+----+-----+--------+---------+-----------+--------+------+----------+

0

In [165]:
# Tratamento de price: padrões (ponto / vírgula / inválido)
df_sem_espacos.select("price").describe().show()

df_preco_padroes = df_sem_espacos.select(
    "price",
    F.when(F.col("price").rlike(r"^\d+(\.\d+)?$"), "ponto")
     .when(F.col("price").rlike(r"^\d+(\,\d+)?$"), "virgula")
     .otherwise("invalido")
     .alias("padrao")
)

# Separar padrões
df_preco_padroes.groupBy(F.col("padrao")).count().show()

# Ver tipos de inválidos
df_preco_padroes.filter(F.col("padrao") == "invalido").select("price").show()

+-------+------------------+
|summary|             price|
+-------+------------------+
|  count|             21613|
|   mean| 540088.1417665294|
| stddev|367127.19648270035|
|    min|      1.00075e+006|
|    max|            999999|
+-------+------------------+

+--------+-----+
|  padrao|count|
+--------+-----+
|invalido| 1492|
|   ponto|20121|
+--------+-----+

+------------+
|       price|
+------------+
|  1.225e+006|
|      2e+006|
|   1.35e+006|
|  1.325e+006|
|   1.04e+006|
|1.09988e+006|
|  1.088e+006|
|   1.45e+006|
|   2.25e+006|
|  1.095e+006|
|  1.505e+006|
|  1.072e+006|
|  1.025e+006|
|    2.4e+006|
|    2.9e+006|
|  1.365e+006|
|   2.05e+006|
|  3.075e+006|
|  2.384e+006|
|  1.384e+006|
+------------+
only showing top 20 rows


In [166]:
# Validação das colunas numéricas (apenas dígitos, com ponto e vírgula)
df_numerico = df_sem_espacos
colunas_selecionadas = [c for c in df_original.columns if c not in ["id", "date", "waterfront", "view", "price", "long"]]

df_numerico.select([
    F.col(c).rlike(r"^\d+([.,]\d+)?$").alias(c) for c in colunas_selecionadas
]).filter(F.col(c) == False).show()

# Validado que as colunas_selecionadas possuem apenas simbologia numérica
# (com ponto e vírgula), sem letras nem caracteres especiais.

+--------+---------+-----------+--------+------+---------+-----+----------+-------------+--------+------------+-------+---+-------------+----------+
|bedrooms|bathrooms|sqft_living|sqft_lot|floors|condition|grade|sqft_above|sqft_basement|yr_built|yr_renovated|zipcode|lat|sqft_living15|sqft_lot15|
+--------+---------+-----------+--------+------+---------+-----+----------+-------------+--------+------------+-------+---+-------------+----------+
+--------+---------+-----------+--------+------+---------+-----+----------+-------------+--------+------------+-------+---+-------------+----------+



In [167]:
# Validação de date via máscara: dígito -> 9, letra -> A
df_colunas_restantes = df_sem_espacos

df_colunas_restantes.select(
    F.regexp_replace(
        F.regexp_replace(F.col("date"), r"[0-9]", "9"),
        r"[A-Za-z]",
        "A"
    ).alias("padrao")
).groupBy(F.col("padrao")).count().orderBy(F.desc("count")).show()

# Somente um padrão encontrado (yyyyMMddThhmmss)

+---------------+-----+
|         padrao|count|
+---------------+-----+
|99999999A999999|21613|
+---------------+-----+



In [168]:
# Validação de waterfront
df_colunas_restantes.groupBy(F.col("waterfront")).count().show()
# Somente 1 ou 0 (considerar true / false)

df_para_cast = df_sem_espacos
df_para_cast = df_para_cast.withColumn("waterfront", F.when(F.col("waterfront") == 1, True).otherwise(False))

# Validação do valor (cast ainda necessário)
df_para_cast.groupBy(F.col("waterfront")).count().show()

+----------+-----+
|waterfront|count|
+----------+-----+
|         0|21450|
|         1|  163|
+----------+-----+

+----------+-----+
|waterfront|count|
+----------+-----+
|      true|  163|
|     false|21450|
+----------+-----+



In [169]:
# Validação de view (padrões)
df_para_cast.groupBy(F.col("view")).count().show()

+----+-----+
|view|count|
+----+-----+
|   3|  510|
|   0|19489|
|   1|  332|
|   4|  319|
|   2|  963|
+----+-----+



In [170]:
# Validação de long: aceita "-" no início, ponto e vírgula; restante apenas dígitos
df_para_cast.select(
    F.col("long").rlike(r"^-?\d+([.,]\d+)?$").alias("long_rlike")
).filter(F.col("long_rlike") != True).show()

+----------+
|long_rlike|
+----------+
+----------+



In [171]:
# Validação de id (apenas dígitos)
df_para_cast.select(
    F.col("id").rlike(r"^\d+$").alias("id")
).filter(F.col("id") != True).show()

+---+
| id|
+---+
+---+



In [172]:
# Conferência: amostra e valores distintos de floors
df_para_cast.show(1)
df_para_cast.select(F.col("floors")).distinct().show()

+----------+---------------+------+--------+---------+-----------+--------+------+----------+----+---------+-----+----------+-------------+--------+------------+-------+-------+--------+-------------+----------+
|        id|           date| price|bedrooms|bathrooms|sqft_living|sqft_lot|floors|waterfront|view|condition|grade|sqft_above|sqft_basement|yr_built|yr_renovated|zipcode|    lat|    long|sqft_living15|sqft_lot15|
+----------+---------------+------+--------+---------+-----------+--------+------+----------+----+---------+-----+----------+-------------+--------+------------+-------+-------+--------+-------------+----------+
|7129300520|20141013T000000|221900|       3|        1|       1180|    5650|     1|     false|   0|        3|    7|      1180|            0|    1955|           0|  98178|47.5112|-122.257|         1340|      5650|
+----------+---------------+------+--------+---------+-----------+--------+------+----------+----+---------+-----+----------+-------------+--------+----

In [ ]:
# Cast final dos tipos (camada silver) — a partir de df_sem_espacos
df_cast = (
    df_sem_espacos
    .withColumn("id", F.col("id").cast(LongType()))
    .withColumn("date", F.to_timestamp("date", "yyyyMMdd'T'HHmmss").cast(TimestampType()))
    .withColumn("price", F.col("price").cast(DoubleType()))
    .withColumn("bathrooms", F.floor(F.col("bathrooms").cast(DoubleType())))
    .withColumn("bedrooms", F.floor(F.col("bedrooms").cast(IntegerType())))
    .withColumn("sqft_living", F.col("sqft_living").cast(DoubleType()))
    .withColumn("sqft_lot", F.col("sqft_lot").cast(DoubleType()))
    .withColumn("floors", F.col("floors").cast(DoubleType()))
    .withColumn("waterfront", F.col("waterfront").cast(BooleanType()))
    .withColumn("view", F.floor(F.col("view").cast(IntegerType())))
    .withColumn("condition", F.col("condition").cast(DoubleType()))
    .withColumn("grade", F.col("grade").cast(DoubleType()))
    .withColumn("sqft_above", F.col("sqft_above").cast(DoubleType()))
    .withColumn("sqft_basement", F.col("sqft_basement").cast(DoubleType()))
    #.withColumn("yr_built", F.col("yr_built").cast(IntegerType()))
    .withColumn("date", F.to_timestamp("date", "yyyy"))
    .withColumn("yr_renovated", F.col("yr_renovated").cast(IntegerType()))
    .withColumn("zipcode", F.col("zipcode").cast(StringType()))
    .withColumn("lat", F.col("lat").cast(DoubleType()))
    .withColumn("long", F.col("long").cast(DoubleType()))
    .withColumn("sqft_living15", F.col("sqft_living15").cast(DoubleType()))
    .withColumn("sqft_lot15", F.col("sqft_lot15").cast(DoubleType()))
)

df_cast.show(5)
df_cast.printSchema()

In [174]:
# Validações — Regra de negócio
# sqft_living     Área útil habitável da casa.
# sqft_lot        Área total do terreno.
# sqft_above      Área habitável acima do nível do solo.
# sqft_basement   Área do porão (subsolo).
# sqft_living15   Média da área habitável das 15 casas vizinhas mais próximas.
# sqft_lot15      Média da área dos terrenos das 15 casas vizinhas mais próximas.

In [175]:
# Floors com parte decimal — inspecionar contra sqft_basement
df_cast.select("id", "floors", "sqft_basement").where(F.col("floors") % 1 != 0).show()
df_cast.select("id", "floors", "sqft_basement").where(F.col("floors") % 1 != 0).count()

+----------+------+-------------+
|        id|floors|sqft_basement|
+----------+------+-------------+
| 114101516|   1.5|          0.0|
|1175000570|   1.5|          0.0|
|6865200140|   1.5|          0.0|
|1202000200|   1.5|          0.0|
|3303700376|   1.5|          0.0|
| 461000390|   1.5|        820.0|
|7589200193|   1.5|          0.0|
|9547205180|   1.5|        790.0|
|4217401195|   1.5|        600.0|
|3253500160|   1.5|       1000.0|
|8820901275|   1.5|        500.0|
|2599001200|   1.5|          0.0|
|1952200240|   1.5|        800.0|
|5200100125|   1.5|        540.0|
| 546000875|   1.5|        500.0|
|1695900060|   1.5|          0.0|
|3524049083|   1.5|        380.0|
|4389200955|   1.5|        770.0|
|1243100136|   1.5|       1490.0|
|1324300398|   1.5|          0.0|
+----------+------+-------------+
only showing top 20 rows


2079

In [176]:
# Regra: se floors tem decimal e não há porão (sqft_basement <= 0), arredonda para baixo;
# caso contrário, mantém o decimal.
df_cast = df_cast.withColumn(
    "floors",
    F.when(
        (F.col("floors") % 1 != 0) & (F.col("sqft_basement") <= 0),
        F.floor(F.col("floors"))
    ).otherwise(F.col("floors"))
)

# Validar aplicação (comparar count com a célula anterior)
df_cast.select(
    "id", "floors", "sqft_basement"
).where(
    (F.col("floors") % 1 != 0) & (F.col("sqft_basement") <= 0)
).show()

+---+------+-------------+
| id|floors|sqft_basement|
+---+------+-------------+
+---+------+-------------+



In [177]:
#Validando coluna price com valor <= 0 
df_cast.select(F.col("price")).where(F.col("price")<=0).show()

#Filtrando dados, com price maior que 0
df_cast = df_cast.filter(F.col("price")> 0)

df_cast.show()

+-----+
|price|
+-----+
+-----+

+----------+-------------------+---------+--------+---------+-----------+--------+------+----------+----+---------+-----+----------+-------------+--------+------------+-------+-------+--------+-------------+----------+
|        id|               date|    price|bedrooms|bathrooms|sqft_living|sqft_lot|floors|waterfront|view|condition|grade|sqft_above|sqft_basement|yr_built|yr_renovated|zipcode|    lat|    long|sqft_living15|sqft_lot15|
+----------+-------------------+---------+--------+---------+-----------+--------+------+----------+----+---------+-----+----------+-------------+--------+------------+-------+-------+--------+-------------+----------+
|7129300520|2014-10-13 00:00:00| 221900.0|       3|        1|     1180.0|  5650.0|   1.0|     false|   0|      3.0|  7.0|    1180.0|          0.0|    1955|           0|  98178|47.5112|-122.257|       1340.0|    5650.0|
|6414100192|2014-12-09 00:00:00| 538000.0|       3|        2|     2570.0|  7242.0|   2.0|  

In [ ]:
# Validando sqft_living (Área útil habitável) e sqft_lot (Área total do terreno)

df_cast.where((F.col("sqft_living") <= 0) | (F.col("sqft_lot") <= 0)).show()

df_cast = df_cast.where((F.col("sqft_living") > 0) & (F.col("sqft_lot") > 0))

df_cast.show()

In [ ]:
# Validando ano de construção, deve ser menor que data da venda
df_cast.where(F.col("yr_built") > F.col("date")).show()

df_cast = df_cast.where(F.col("yr_built") <= F.col("date"))

df_cast.show()

In [ ]:
# Validando ano de renovação
df_cast.where((F.col("yr_renovated") < F.col("yr_built")) & (F.col("yr_renovated") > 0)).show()

df_cast = df_cast.where((F.col("yr_renovated") < F.col("yr_built")) & (F.col("yr_renovated") > 0))